# Projeto SETA - Classificador supervisionado


### 1) Qual a base escolhida?

**Base escolhida:** *Hotel Booking Demand* (`hotel_bookings.csv`). **Objetivo:** Prever o cancelamento de reservas hoteleiras (problema de classificação binária supervisionada)

### 2) **(10%)** Pré-processamento: entendimento do conjunto de dados
* **Quais são minhas features?** As variáveis independentes (preditoras) representam as características da reserva, como `lead_time`, `stays_in_weekend_nights`, `stays_in_week_nights`, `adults`, `children`, `meal`, `market_segment`, `deposit_type`, `customer_type`, `adr`, entre outras.

* **Quais são minhas classes?** A variável alvo (target) é **is\_canceled**:
* **0**: Reserva realizada e mantida (Não Cancelada).
* **1**: Reserva cancelada pelo cliente.

* **Como estão distribuídas minhas classes?** Aproximadamente 63% das reservas são da classe `0` (Não Cancelada) e 37% da classe `1` (Cancelada).

* **Checagem de limites e tratamento de valores ausentes:**
* Remoção de colunas com excesso de valores nulos ou identificadores irrelevantes (`company`, `agent`, `reservation_status_date`).
* Imputação da mediana para valores numéricos nulos e da moda para variáveis categóricas.
* Remoção da coluna `reservation_status` para evitar vazamento de dados (*data leakage*).

* **Conversão do tipo de dados:** Aplicação de *One-Hot Encoding* (`pd.get_dummies`) para converter variáveis categóricas de texto em colunas numéricas binárias.


Pré-processamento e Divisão Treino/Teste

---



In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
     confusion_matrix, classification_report, accuracy_score,
     precision_score, recall_score, f1_score
     )

# 1. Carregar dataset
df = pd.read_csv('hotel_bookings.csv')

# 2. Limpeza inicial
colunas_remover = ['company', 'agent', 'reservation_status_date', 'reservation_status']
df = df.drop(columns=[c for c in colunas_remover if c in df.columns])

# Tratar nulos
# Tratar nulos de forma segura testando se a coluna é numérica 
for col in df.columns: 
  if pd.api.types.is_numeric_dtype(df[col]): 
    df[col] = df[col].fillna(df[col].median()) 
  else: 
    if not df[col].mode().empty: 
      df[col] = df[col].fillna(df[col].mode()[0])

# 3. Separar X e y
X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

# Codificação Dummy
X = pd.get_dummies(X, drop_first=True)

# 4. Divisão Treino (70%) e Teste (30%) com Estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
    )

print(f"Treino X: {X_train.shape} | Teste X: {X_test.shape}")

Treino X: (83573, 245) | Teste X: (35817, 245)


### 3) **(80%)** Nos blocos seguintes implemente seus classificadores (serão implementados 2 métodos diferentes).

#### 3.1) Qual método escolhido?

**Indique o método escolhido**

**Método 1:** **K-Nearest Neighbors (KNN)**

#### 3.2) **(10%)** Baseline - Implemente seu classificador da forma mais simples possível para esse ser seu baseline

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# KNN Baseline (Sem padronização, K=5 padrão)
knn_baseline = KNeighborsClassifier(n_neighbors=5)
knn_baseline.fit(X_train, y_train)

y_pred_knn_base = knn_baseline.predict(X_test)

print("=== KNN Baseline ===")
print(classification_report(y_test, y_pred_knn_base))

=== KNN Baseline ===
              precision    recall  f1-score   support

           0       0.80      0.85      0.82     22550
           1       0.71      0.63      0.67     13267

    accuracy                           0.77     35817
   macro avg       0.75      0.74      0.74     35817
weighted avg       0.76      0.77      0.76     35817



#### 3.3) **(20%)** Versão 1 - O que podemos fazer para melhorar nosso baseline? Aplique técnicas como redução de dimensionalidade, normalização ou outras. Compare os resultados.

In [3]:
# Escalonamento das variáveis (fundamental para algoritmos de distância como KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_v1 = KNeighborsClassifier(n_neighbors=5)
knn_v1.fit(X_train_scaled, y_train)

y_pred_knn_v1 = knn_v1.predict(X_test_scaled)

print("=== KNN Versão 1 (Com Normalização) ===")
print(classification_report(y_test, y_pred_knn_v1))

=== KNN Versão 1 (Com Normalização) ===
              precision    recall  f1-score   support

           0       0.85      0.87      0.86     22550
           1       0.77      0.74      0.76     13267

    accuracy                           0.82     35817
   macro avg       0.81      0.81      0.81     35817
weighted avg       0.82      0.82      0.82     35817



#### 3.4) **(10%)** Tunning - Agora que temos um resultado promissor, vamos tentar melhorar o resultado alterando um ou mais hiper-parametro. Compare os resultados.

In [4]:
# Testando valores de K para encontrar o melhor desempenho
k_values = [3, 7, 11, 15]
best_k = 5
best_f1 = 0

for k in k_values:
  knn_temp = KNeighborsClassifier(n_neighbors=k)
  knn_temp.fit(X_train_scaled, y_train)
  preds = knn_temp.predict(X_test_scaled)
  f1 = f1_score(y_test, preds)
  print(f"K = {k} -> F1-Score: {f1:.4f}")
  if f1 > best_f1:
    best_f1 = f1
    best_k = k

print(f"\nMelhor K encontrado: {best_k}")

# Modelo final Tunado
knn_tuned = KNeighborsClassifier(n_neighbors=best_k)
knn_tuned.fit(X_train_scaled, y_train)
y_pred_knn_tuned = knn_tuned.predict(X_test_scaled)

K = 3 -> F1-Score: 0.7613
K = 7 -> F1-Score: 0.7539
K = 11 -> F1-Score: 0.7499
K = 15 -> F1-Score: 0.7403

Melhor K encontrado: 3


#### 3.5) Qual método escolhido?

**Método 2:** **Árvore de Decisão (Decision Tree Classifier)**

#### 3.6) **(10%)** Baseline - Implemente seu classificador da forma mais simples possível para esse ser seu baseline

In [5]:
# Árvore de Decisão sem limite de profundidade (propenso a overfitting)
dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train, y_train)

y_pred_dt_base = dt_baseline.predict(X_test)

print("=== Decision Tree Baseline ===")
print(classification_report(y_test, y_pred_dt_base))

=== Decision Tree Baseline ===
              precision    recall  f1-score   support

           0       0.88      0.88      0.88     22550
           1       0.79      0.80      0.80     13267

    accuracy                           0.85     35817
   macro avg       0.84      0.84      0.84     35817
weighted avg       0.85      0.85      0.85     35817



#### 3.7) **(20%)** Versão 1 - O que podemos fazer para melhorar nosso baseline? Aplique técnicas como redução de dimensionalidade, normalização ou outras. Compare os resultados.

In [6]:
# Definindo critério de entropia e profundidade máxima inicial para controlar overfitting
dt_v1 = DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=42)
dt_v1.fit(X_train, y_train)

y_pred_dt_v1 = dt_v1.predict(X_test)

print("=== Decision Tree Versão 1 ===")
print(classification_report(y_test, y_pred_dt_v1))

=== Decision Tree Versão 1 ===
              precision    recall  f1-score   support

           0       0.85      0.90      0.87     22550
           1       0.81      0.73      0.77     13267

    accuracy                           0.84     35817
   macro avg       0.83      0.81      0.82     35817
weighted avg       0.83      0.84      0.83     35817



#### 3.8) **(10%)** Tunning - Agora que temos um resultado promissor, vamos tentar melhorar o resultado alterando um ou mais hiper-parametro. Compare os resultados.

In [7]:
# Testando variações de profundidade máxima e amostras mínimas por folha
depths = [8, 12, 16, 20]
best_depth = 8
best_f1_dt = 0

for d in depths:
  dt_temp = DecisionTreeClassifier(max_depth=d, min_samples_split=10, random_state=42)
  dt_temp.fit(X_train, y_train)
  preds = dt_temp.predict(X_test)
  f1 = f1_score(y_test, preds)
  print(f"Max Depth = {d} -&gt; F1-Score: {f1:.4f}")
  if f1 > best_f1_dt:
    best_f1_dt = f1
    best_depth = d

print(f"\nMelhor Max Depth: {best_depth}")

dt_tuned = DecisionTreeClassifier(max_depth=best_depth, min_samples_split=10, random_state=42)
dt_tuned.fit(X_train, y_train)
y_pred_dt_tuned = dt_tuned.predict(X_test)

Max Depth = 8 -&gt; F1-Score: 0.7511
Max Depth = 12 -&gt; F1-Score: 0.7779
Max Depth = 16 -&gt; F1-Score: 0.7954
Max Depth = 20 -&gt; F1-Score: 0.7963

Melhor Max Depth: 20


### 5) **(10%)** Conclusões

*Compare seus resultados. Imaginando que sua solução fosse para produção, qual deles você escolheria? Por que? Quais os riscos você enxerga? O que recomendaria de próximos passos para melhorar os resultados?*

5) Conclusões

1. Comparação dos Resultados e Desempenho dos Modelos

A avaliação dos modelos supervisionados para previsão de cancelamento de reservas (is_canceled) apresentou a seguinte evolução:
* Método 1 - K-Nearest Neighbors (KNN):

  - Baseline (K=5 padrão, sem escala): Acurácia de 77% e F1-Score de 0,67 na classe de cancelamento (1).
  - Versão 1 (Com StandardScaler): A acurácia subiu para 82% e o F1-Score da classe alvo aumentou para 0,76\. A normalização foi crucial, pois o KNN calcula distâncias euclidianas, sendo afetado pelas diferenças de escala entre as 245 variáveis.
  - Tuning (Ajuste do hiperparâmetro K): Dentre os valores testados (K = 3, 7, 11, 15), o melhor resultado foi obtido com K = 3, alcançando F1-Score de 76,13%.
* Método 2 - Árvore de Decisão (Decision Tree Classifier):
  - Baseline (Profundidade livre): Obteve acurácia de 85% e F1-Score de 0,80 na classe de cancelamento (1).
  - Versão 1 (criterion='entropy', max_depth=10): Apresentou acurácia de 84% e F1-Score de 0,77.
  - Tuning (Ajuste de max_depth e min_samples_split): Testando profundidades de 8 a 20, o modelo com max\_depth = 20 obteve a melhor performance, atingindo F1-Score de 79,63% na classe 1 e acurácia geral de 85%.
  
---

2. Escolha do Modelo para Produção
Recomendamos a Árvore de Decisão (Versão Tunada com max_depth = 20) para implantação em produção.

Justificativas:

1. Desempenho Superior: Apresentou a maior acurácia geral (85%) e o maior F1-Score na classe de cancelamento (79,63%), garantindo melhor identificação de reservas de risco.

2. Eficiência Computacional Extrema: A Árvore de Decisão levou apenas 9 segundos para executar o tuning e realizar inferências, enquanto o KNN levou cerca de 5 minutos na busca de vizinhos. Em produção com grande volume de dados, o tempo de resposta e o custo de infraestrutura da Árvore são significativamente menores.
3. Interpretabilidade de Negócio: Permite extrair a relevância das variáveis (como deposit\_type, lead\_time e adr), fornecendo justificativas claras aos gestores do hotel sobre os motivos que levam ao cancelamento.

---

3. Riscos Identificados
- Mudança no Padrão de Cancelamentos (Data Drift): Alterações econômicas, novas políticas de viagem ou fatores sazonais podem alterar a distribuição dos dados de produção em relação à base de treino.
- Overfitting Residual: Profundidades elevadas (como max\_depth = 20) podem memorizar especificidades da base histórica se não houver um retreinamento periódico com novos dados.

---

4. Próximos Passos

  1\. Testar Algoritmos Ensemble: Implementar Random Forest ou XGBoost para combinar múltiplas árvores e aumentar a capacidade preditiva reduzindo a variação.

  2\. Implementar Pipeline de MLOps: Estruturar monitoramento contínuo do F1-Score em produção e rotinas de retreinamento automatizado.